<div style="width: 150px; float: left;"> <img src="https://raw.githubusercontent.com/DataForScience/Networks/master/data/D4Sci_logo_ball.png" alt="Data For Science, Inc" align="left" border="0" width=150px> </div>
    <div style="float: left; margin-left: 10px;"> <h1>Automate The Boring Stuff With LLMs</h1>
<h1>Summarization: arXiv Triage & Action-Items</h1>
        <p>Bruno Gonçalves<br/>
        <a href="http://www.data4sci.com/">www.data4sci.com</a><br/>
            @bgoncalves, @data4sci</p></div>
</div>

In [1]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

from pprint import pprint

import matplotlib
import matplotlib.pyplot as plt

import pandas as pd
import numpy as np

from tqdm.notebook import tqdm
tqdm.pandas()

from typing import List, Literal
from pydantic import BaseModel, Field

from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableLambda

from datasets import load_dataset

import watermark

%load_ext watermark
%matplotlib inline

We start by print out the versions of the libraries we're using for future reference

In [2]:
%watermark -n -v -m -g -iv

Python implementation: CPython
Python version       : 3.14.5
IPython version      : 9.17.1

Compiler    : Clang 22.1.3 
OS          : Darwin
Release     : 25.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 16
Architecture: 64bit

Git hash: a5b9a638f83c6ba698ba0a0325aa81d463ad4d4c

datasets                : 5.1.0
json                    : 2.0.9
langchain_anthropic     : 1.7.6
langchain_core          : 1.6.10
langchain_text_splitters: 1.1.3
matplotlib              : 3.11.2
numpy                   : 2.5.4
pandas                  : 3.0.6
pydantic                : 2.14.1
tqdm                    : 4.70.1
watermark               : 2.6.0



Load default figure style

In [3]:
plt.style.use("d4sci.mplstyle")
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# Setup & data

In [4]:
llm = ChatAnthropic(
        model='claude-opus-4-5', 
        temperature=0, 
        max_tokens=2048
    )

Streaming download — the full set is large, just pull a slice.

In [5]:
ds = load_dataset('ccdv/arxiv-summarization', 'document', split='train',
                  streaming=True)
papers = []
for i, ex in enumerate(ds):
    if i >= 200:
        break
    papers.append(ex)

df = pd.DataFrame(papers)
df['n_chars'] = df['article'].str.len()

# Keep medium-to-long papers — that's where map-reduce earns its keep.
df = df[(df['n_chars'] > 8000) & (df['n_chars'] < 60000)].head(100).reset_index(drop=True)
print('Working set:', df.shape)
df[['n_chars']].describe().T

Working set: (100, 3)


,count,mean,std,min,25%,50%,75%,max
n_chars,100.0,28483.82,12110.956944,8984.0,19338.25,26102.0,36539.25,53839.0


# Action-Item template (structured output)

In [6]:
class ActionItem(BaseModel):
    action: Literal['read-cited-paper', 'replicate-experiment', 'try-method',
                    'flag-for-review', 'discuss-with-team', 'no-action']
    detail: str = Field(..., description='Concrete next step, < 200 chars')
    priority: Literal['low', 'medium', 'high']

class PaperBrief(BaseModel):
    summary: str
    action_items: List[ActionItem] = Field(default_factory=list, max_length=5)
    open_questions: List[str] = Field(default_factory=list, max_length=3)

brief_parser = PydanticOutputParser(pydantic_object=PaperBrief)

BRIEF_PROMPT = ChatPromptTemplate.from_messages([
    ('system', 'You produce researcher briefs from paper summaries. '
               'Be concrete: "replicate Table 3 results on CIFAR-10" beats '
               '"investigate further".\n\n{format_instructions}'),
    ('user', 'Paper executive summary:\n{summary}'),
]).partial(format_instructions=brief_parser.get_format_instructions())

In [7]:
pprint(BRIEF_PROMPT.partial_variables['format_instructions'])

('The output should be formatted as a JSON instance that conforms to the JSON '
 'schema below.\n'
 '\n'
 'As an example, for the schema {"properties": {"foo": {"title": "Foo", '
 '"description": "a list of strings", "type": "array", "items": {"type": '
 '"string"}}}, "required": ["foo"]}\n'
 'the object {"foo": ["bar", "baz"]} is a well-formatted instance of the '
 'schema. The object {"properties": {"foo": ["bar", "baz"]}} is not '
 'well-formatted.\n'
 '\n'
 'Here is the output schema:\n'
 '```\n'
 '{"$defs": {"ActionItem": {"properties": {"action": {"enum": '
 '["read-cited-paper", "replicate-experiment", "try-method", '
 '"flag-for-review", "discuss-with-team", "no-action"], "title": "Action", '
 '"type": "string"}, "detail": {"description": "Concrete next step, < 200 '
 'chars", "title": "Detail", "type": "string"}, "priority": {"enum": ["low", '
 '"medium", "high"], "title": "Priority", "type": "string"}}, "required": '
 '["action", "detail", "priority"], "title": "ActionItem", 

In [8]:
summary = """# Executive Summary

This paper addresses the challenge of achieving fast learning rates for support vector machines (SVMs) in high-dimensional settings by 
investigating whether additive kernel structures can overcome the curse of dimensionality that plagues general nonparametric methods. 
The authors develop a theoretical framework for SVMs with additive kernels using general convex Lipschitz continuous loss functions 
(particularly pinball loss for quantile regression), establishing novel approximation error bounds and proving that covering numbers 
for additive hypothesis spaces have a capacity parameter independent of the number of input dimensions—a key technical contribution. 
The main result demonstrates that when the additive model assumption holds, SVMs with additive kernels achieve substantially faster 
learning rates than SVMs with general kernels (such as Gaussian RBF) in high dimensions, with quantile regression rates approaching 
n^(-1/2) that are optimal in the SVM literature and independent of both dimension and the number of additive components. A limitation 
is that the approach requires the true target function to satisfy the additive structure assumption, and for small dimensions (d=1 or d=2),
the rates may not improve over standard methods; additionally, establishing whether specific target functions satisfy the required 
approximation error conditions remains an open problem. This work matters because it provides theoretical justification for using 
additive models in high-dimensional machine learning applications where interpretability is important, demonstrating that practitioners 
can achieve both the flexibility of nonparametric methods and protection against dimensional scaling issues when additive structure is appropriate.
"""

In [9]:
brief_chain = BRIEF_PROMPT | llm | brief_parser

brief = brief_chain.invoke({'summary': summary})
print(brief.model_dump_json(indent=2))

{
  "summary": "This paper proves that SVMs with additive kernels can escape the curse of dimensionality when the true target function has additive structure. The key technical contribution shows covering numbers for additive hypothesis spaces have capacity independent of input dimension. For quantile regression with pinball loss, they achieve near-optimal n^(-1/2) rates independent of dimension d and number of additive components. The catch: the additive assumption must hold, and benefits only appear for d≥3.",
  "action_items": [
    {
      "action": "try-method",
      "detail": "Test additive kernel SVM vs Gaussian RBF SVM on a high-dimensional dataset (d≥10) where additive structure is plausible (e.g., GAM-style regression problems)",
      "priority": "high"
    },
    {
      "action": "read-cited-paper",
      "detail": "Read the cited work on covering number bounds for additive spaces to understand the dimension-independence proof technique",
      "priority": "medium"
    },

# Router prompt

Classify each paper into a category, then dispatch to a specialist summarizer. We'll keep two specialists: `theory` (proof-heavy, focus on assumptions and lemmas) and `applied` (focus on dataset, metric, baseline, delta).

Why bother? A theory paper summarised with an 'applied' template often invents fake datasets and metrics. The router prevents that whole class of error.

In [10]:
# Shared rubric used by *both* the router and the auditor so they answer
# the same question. The router's job is to triage; the auditor's job is to
# catch routing mistakes by re-asking with the same definitions.
LABELS = ['theory', 'experimental', 'survey']

LABEL_RUBRIC = """\
Choose exactly one label using these rules, in order:

1. survey  - The paper's primary contribution is to *review, taxonomize, or
   compare* prior work. Look for phrases like "we review", "in this survey",
   long reference lists with little new analysis, or section headings that
   organize the literature. If a paper presents new theory or experiments only
   in passing, it is NOT a survey.

2. theory  - The paper's primary contribution is a *mathematical or formal
   result*: a theorem, proof, derivation, model, bound, convergence rate, or
   analytical framework, with no new physical measurements or empirical data.
   Pure simulations of an analytical model still count as theory if the
   contribution is the model.

3. experimental - The paper's primary contribution is *empirical evidence*:
   measurements from an instrument, observations, lab experiments, numerical
   experiments on real data, or benchmark results. This includes physics
   experiments, astronomical observations, and machine-learning evaluations.

When a paper mixes types, pick the label that matches the *headline
contribution* (usually what the abstract emphasizes most). When unsure between
theory and experimental, prefer theory only if the central novelty is the
proof/derivation itself."""

ROUTER_FEW_SHOTS = [
    ('...we prove a non-asymptotic convergence rate of O(n^-1/2) for stochastic '
     'gradient descent on strongly convex objectives...',
     'theory'),
    ('We report Josephson tunneling measurements in Nb/InAs/Nb junctions at 20 mK, '
     'observing a 4pi-periodic current-phase relation consistent with Majorana modes...',
     'experimental'),
    ('This paper surveys recent advances in graph neural networks, organizing methods '
     'into spectral, spatial, and attention-based families and identifying open gaps...',
     'survey'),
]

FEW_SHOT_BLOCK = '\n'.join(f'ABSTRACT: {a}\nLABEL: {l}' for a, l in ROUTER_FEW_SHOTS)

def parse_label(raw: str) -> str:
    """Normalise a model output to one of LABELS. 'applied' is aliased to 'experimental'."""
    
    word = ''.join(c for c in raw.strip().lower() if c.isalpha())
    
    if word == 'applied':
        word = 'experimental'

    return word if word in LABELS else 'experimental'

# Cap the cheap router/audit calls at a few tokens so they can't ramble.
router_llm = llm.bind(max_tokens=4)

In [11]:
ROUTER = ChatPromptTemplate.from_messages([
    ('system',
     'You are a careful arXiv triage classifier.\n\n'
     f'{LABEL_RUBRIC}\n\n'
     'Examples:\n' + FEW_SHOT_BLOCK +
     '\n\nRespond with ONLY the single lowercase label, no punctuation.'),
    ('user', 'ABSTRACT: {abstract}\nLABEL:'),
])
router_chain = ROUTER | router_llm | StrOutputParser()

THEORY = ChatPromptTemplate.from_messages([
    ('system',
     'Summarize the THEORETICAL contribution of this paper. Cover: '
     '(1) setting and assumptions, '
     '(2) main theorem or formal result, '
     '(3) proof technique, '
     '(4) where the bound or argument is tight, '
     '(5) one limitation. 4-5 sentences. '
     "Start your reply with exactly: 'Category: theory.'"),
    ('user', '{summary}'),
])
EXPERIMENTAL = ChatPromptTemplate.from_messages([
    ('system',
     'Summarize the EMPIRICAL contribution of this paper. Cover: '
     '(1) phenomenon or task studied,' 
     '(2) system / dataset / instrument, '
     '(3) method or measurement protocol, '
     '(4) headline quantitative result, '
     '(5) one limitation. 4-5 sentences. '
     "Start your reply with exactly: 'Category: experimental.'"),
    ('user', '{summary}'),
])
SURVEY = ChatPromptTemplate.from_messages([
    ('system',
     'Summarize the SURVEY contribution of this paper. Cover: '
     '(1) scope and motivation, '
     '(2) taxonomy proposed, '
     '(3) ~3 representative works cited, '
     '(4) the gap or open problem highlighted, '
     '(5) one limitation of the survey itself. 4-5 sentences. '
     "Start your reply with exactly: 'Category: survey.'"),
    ('user', '{summary}'),
])

specialists = {
    'theory': THEORY | llm | StrOutputParser(),
    'experimental': EXPERIMENTAL | llm | StrOutputParser(),
    'survey': SURVEY | llm | StrOutputParser(),
}

def summarize(text: str) -> str:    
    return brief_chain.invoke({'summary': text})


def route_and_summarize(paper_text: str) -> dict:
    abstract = paper_text[:2000]
    category = parse_label(router_chain.invoke({'abstract': abstract}))

    base_summary = summarize(paper_text)
    final = specialists[category].invoke({'summary': base_summary})

    return {'category': category, 'summary': final}

Run on a small batch with progress

In [12]:
outputs = []
for txt in tqdm(df['article'].head(5), desc='Triage', total=5):
    try:
        outputs.append(route_and_summarize(txt))
    except Exception as e:
        outputs.append({'category': 'error', 'summary': str(e)[:200]})

pd.DataFrame(outputs).head()

Triage:   0%|          | 0/5 [00:00<?, ?it/s]

,category,summary
0,theory,Category: theory.\n\nThis paper analyzes SVMs ...
1,experimental,Category: experimental.\n\nThis CLEO-c paper m...
2,theory,Category: theory.\n\nThis paper examines trans...
3,theory,Category: theory.\n\nThis paper analyzes scint...
4,theory,Category: theory.\n\nI cannot provide a theore...


In [14]:
print(outputs[1]["summary"])

Category: experimental.

This CLEO-c paper measures the leptonic decay branching fraction B(D_s+ → τ+ν_τ) using 586 pb⁻¹ of e+e- collision data collected at √s = 4170 MeV, employing a double-tag technique where D_s mesons are reconstructed in three hadronic modes (K+K-π+, K_S K+, ηπ+) opposite the signal decay with τ+ → e+ν_e ν̄_τ. They obtain B(D_s+ → τ+ν_τ) = (5.30 ± 0.47 ± 0.22)% and extract the decay constant f_Ds = 275 ± 13 MeV, which when combined with companion measurements yields f_Ds = 274 ± 10 MeV—the most precise determination at that time. This experimental value exceeds the HPQCD lattice QCD prediction of 241 ± 3 MeV by 3.3 standard deviations, potentially indicating physics beyond the Standard Model. A key limitation is that the analysis relies on external inputs for D_s production cross-sections and cannot independently constrain all systematic uncertainties associated with the tag-mode reconstruction efficiencies.


<center>
     <img src="data/D4Sci_logo_full.png" alt="Data For Science, Inc" align="center" border="0" width=300px> 
</center>